# 02 — Preprocessing and feature engineering
Target is Appliances at the next observed row. Rolling predictors are shifted to use history only.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src' / 'wattwise').exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))


In [2]:
from wattwise.data_loader import load_data
from wattwise.preprocessing import clean_data
from wattwise.feature_engineering import make_features,chronological_split
raw=load_data(); clean,report=clean_data(raw); report

{'before': {'rows': 19735, 'missing': 0, 'duplicates': 0},
 'after': {'rows': 19735, 'missing': 0, 'duplicates': 0},
 'leading_rows_dropped_without_history': 0,
 'decisions': 'Chronological ordering; duplicate timestamps keep last; numeric missing values are forward-filled from past only, and leading rows without history are dropped. Extremes are retained as potentially real use events.'}

In [3]:
X,y,t=make_features(clean)
X.shape, y.head(), t.head(), X.columns.tolist()

((19590, 63),
 144     60.0
 145    210.0
 146    380.0
 147    370.0
 148    230.0
 Name: target_next, dtype: float64,
 144   2016-01-12 17:00:00
 145   2016-01-12 17:10:00
 146   2016-01-12 17:20:00
 147   2016-01-12 17:30:00
 148   2016-01-12 17:40:00
 Name: date, dtype: datetime64[us],
 ['lights',
  'T1',
  'RH_1',
  'T2',
  'RH_2',
  'T3',
  'RH_3',
  'T4',
  'RH_4',
  'T5',
  'RH_5',
  'T6',
  'RH_6',
  'T7',
  'RH_7',
  'T8',
  'RH_8',
  'T9',
  'RH_9',
  'T_out',
  'Press_mm_hg',
  'RH_out',
  'Windspeed',
  'Visibility',
  'Tdewpoint',
  'rv1',
  'rv2',
  'appliances_current',
  'appliances_lag_1',
  'appliances_lag_2',
  'appliances_lag_3',
  'appliances_lag_6',
  'appliances_lag_12',
  'appliances_lag_144',
  'appliances_roll_mean_3',
  'appliances_roll_std_3',
  'appliances_roll_mean_6',
  'appliances_roll_std_6',
  'appliances_roll_mean_12',
  'appliances_roll_std_12',
  'appliances_roll_mean_144',
  'appliances_roll_std_144',
  'appliances_change_1',
  'lights_change_1',


Rows without historical warm-up or a next observed target are removed. Fit any learned preprocessing on training only.

In [4]:
parts=chronological_split(X,y,t)
[(k,len(v[0]),v[2].min(),v[2].max()) for k,v in parts.items()]

[('train',
  13713,
  Timestamp('2016-01-12 17:00:00'),
  Timestamp('2016-04-16 22:20:00')),
 ('validation',
  2938,
  Timestamp('2016-04-16 22:30:00'),
  Timestamp('2016-05-07 08:00:00')),
 ('test',
  2939,
  Timestamp('2016-05-07 08:10:00'),
  Timestamp('2016-05-27 17:50:00'))]

In [5]:
lag_cols=[c for c in X if c.startswith('appliances_lag_') or 'roll_' in c]
sensor_cols=[c for c in X if c not in lag_cols]
len(lag_cols),len(sensor_cols)

(14, 49)

## Summary
The split stays chronological. High values are retained for investigation. To quantify feature-group value, fit each configuration on training history and compare validation metrics only.

## Cleaning report and decisions

Inspect counts before and after cleaning. Duplicate timestamps keep the last reading; numeric gaps are forward-filled from past observations only and leading rows without history are dropped. In the downloaded source there are no missing cells, so imputation does not change this dataset. Extremes remain in place.

In [6]:
report, clean.shape


({'before': {'rows': 19735, 'missing': 0, 'duplicates': 0},
  'after': {'rows': 19735, 'missing': 0, 'duplicates': 0},
  'leading_rows_dropped_without_history': 0,
  'decisions': 'Chronological ordering; duplicate timestamps keep last; numeric missing values are forward-filled from past only, and leading rows without history are dropped. Extremes are retained as potentially real use events.'},
 (19735, 29))

## Target alignment example

For each row *t*, the target is the next observed Appliances value. The feature matrix contains current Appliance consumption and historical lags. The displayed rows make the time alignment concrete.

In [7]:
alignment=clean[['date','Appliances']].copy()
alignment['target_next']=alignment.Appliances.shift(-1)
alignment['next_timestamp']=alignment.date.shift(-1)
alignment[['date','Appliances','next_timestamp','target_next']].head(8)


,date,Appliances,next_timestamp,target_next
0,2016-01-11 17:00:00,60,2016-01-11 17:10:00,60.0
1,2016-01-11 17:10:00,60,2016-01-11 17:20:00,50.0
2,2016-01-11 17:20:00,50,2016-01-11 17:30:00,50.0
3,2016-01-11 17:30:00,50,2016-01-11 17:40:00,60.0
4,2016-01-11 17:40:00,60,2016-01-11 17:50:00,50.0
5,2016-01-11 17:50:00,50,2016-01-11 18:00:00,60.0
6,2016-01-11 18:00:00,60,2016-01-11 18:10:00,60.0
7,2016-01-11 18:10:00,60,2016-01-11 18:20:00,60.0


## Feature definitions and availability

Every predictor must be available at the forecast origin. Rolling features shift the consumption series before aggregation, preventing the value being predicted from entering its own history summary. The model matrix also adds past-only appliance/sensor changes and weekly cyclic time terms. The feature set also adds one-step changes in Appliance, lights, and selected indoor/outdoor temperature, humidity, and weather sensors. Weekly sine/cosine terms encode recurring week position without a discontinuous Sunday-to-Monday boundary. These past-only engineered features increase the model matrix to 63 columns.

In [8]:
import pandas as pd
feature_notes={'appliances_current':'Appliances observed at t, Wh/10 min','appliances_lag_k':'Appliances k observations before t','appliances_roll_mean_w':'mean of the preceding w readings, excludes t','appliances_roll_std_w':'standard deviation of the preceding w readings, excludes t','hour_sin/hour_cos':'cyclic encoding of current hour','day_of_week/is_weekend':'calendar fields at t','target_next':'Appliances at next observed row; target only'}
pd.Series(feature_notes,name='definition').to_frame()


,definition
appliances_current,"Appliances observed at t, Wh/10 min"
appliances_lag_k,Appliances k observations before t
appliances_roll_mean_w,"mean of the preceding w readings, excludes t"
appliances_roll_std_w,standard deviation of the preceding w readings...
hour_sin/hour_cos,cyclic encoding of current hour
day_of_week/is_weekend,calendar fields at t
target_next,Appliances at next observed row; target only


## Leakage checks

Confirm that the last engineered feature row corresponds to a timestamp before the final raw row, that next-target values are shifted, and that each split occupies a later time block. The model's learned transformations are fit only on training rows.

In [9]:
assert (t.reset_index(drop=True) < clean.date.iloc[-1]).all()
assert X.index.is_monotonic_increasing
assert parts['train'][2].max() < parts['validation'][2].min() < parts['test'][2].min()
{'first_feature_time':str(t.iloc[0]),'last_feature_time':str(t.iloc[-1]),'last_raw_time':str(clean.date.iloc[-1]),'chronological_order':True}


{'first_feature_time': '2016-01-12 17:00:00',
 'last_feature_time': '2016-05-27 17:50:00',
 'last_raw_time': '2016-05-27 18:00:00',
 'chronological_order': True}

## Feature-group ablation

The experiment fits on the training block and compares validation MAE/RMSE/R². Mutual information ranks are calculated from training only. The test period is not used in this comparison.

In [10]:
from pathlib import Path
import pandas as pd
abl=Path('reports/metrics/feature_ablation.csv')
pd.read_csv(abl) if abl.exists() else 'Run python -m wattwise.train to generate this train/validation ablation.'


,feature_set,feature_count,MAE,RMSE,R2,n_obs,run_id
0,all_features,63,22.121437,57.324509,0.596166,2938,55004ee9-e3d3-433c-9528-6e2227d2b7d8
1,without_consumption_history,49,22.265324,57.473067,0.594070,2938,55004ee9-e3d3-433c-9528-6e2227d2b7d8
2,mutual_information_top_20,20,22.012251,57.263843,0.597021,2938,55004ee9-e3d3-433c-9528-6e2227d2b7d8


## Preprocessing conclusion

Describe how many rows were removed by warm-up and next-target alignment, the final feature count, and whether selected history features improve validation performance. Explain that the chronological holdout remains isolated from these choices.

In [11]:
print('Usable rows:',len(X),'features:',X.shape[1])
print('Split sizes:',{k:len(v[0]) for k,v in parts.items()})


Usable rows: 19590 features: 63
Split sizes: {'train': 13713, 'validation': 2938, 'test': 2939}
